# CAP 4611 — Assignment 3: Classification (Telco Customer Churn)
## Phase 1 Submission — Sections A through F

`This jupyter notebook is prepared by "Richard Magiday".`

> Fill in all code cells and markdown discussion cells as instructed.

<font color='red'>Phase 1 due date: July 7 — submit this PDF containing solutions (including outputs) to sections A through F. Failure to submit will result in a 10% penalty on the total assignment grade.</font>

## A. Load Data and Perform Basic EDA [10 pts = 1 + 1 + 1 + 2 + 5]

### I. **Import libraries:** pandas, numpy, matplotlib (set `%matplotlib inline`), matplotlib's pyplot, seaborn, missingno, scipy's stats, sklearn. (1 pt)

In [ ]:
# Install any packages that may be missing in this environment (safe no-op if already installed)
!pip install -q missingno imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import missingno as msno
from scipy import stats
import sklearn

sns.set_style('whitegrid')

### II. Load the dataset and display the number of rows and columns. (1 pt)

In [ ]:
df = pd.read_csv('telco_churn2.csv')
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

### III. Display the first 5 and last 5 rows. (1 pt)

In [ ]:
display(df.head())
display(df.tail())

### IV. Show how many columns contain null values. (2 pts)

In [ ]:
null_counts = df.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]
print(f"Number of columns containing null values: {len(cols_with_nulls)}")
print(cols_with_nulls)

msno.bar(df)
plt.show()

### V. Plot the target (`Churn`) distribution. In a markdown cell, discuss the class imbalance, what issues it may cause in your models, and propose at least one solution. (5 pts)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Churn', data=df)
plt.title('Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.show()

print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).mul(100).round(2))

The dataset is imbalanced: about 5,174 customers (73.5%) did not churn ("No") while only 1,869 (26.5%) churned ("Yes"), roughly a 3:1 ratio. If we train a classifier directly on this data, it can achieve high accuracy simply by predicting the majority class ("No") most of the time, while performing poorly at recognizing actual churners (low recall/precision on the "Yes" class) — which is the class the company actually cares about identifying. This can also bias the decision boundary of algorithms like logistic regression, SVM, or KNN toward the majority class. A common solution is to resample the training data, e.g., using SMOTE (Synthetic Minority Oversampling Technique) to oversample the minority ("Yes") class, or to undersample the majority class, or to use `class_weight='balanced'` in models that support it. We apply SMOTE to the training set in Section C. We should also evaluate models using metrics beyond accuracy, such as precision, recall, F1-score, and ROC-AUC, which are more informative for imbalanced classification tasks.

## B. Feature Selection and Pre-processing [32 pts]

### I. Drop identifier and constant columns (4 pts)
Look at the number of unique values in `customerID`, `SupportTicketID`, and `ActiveCountryCode`. Two of them are essentially unique per row and one is constant for every row. In a markdown cell, explain why each one carries no predictive value, then drop all three. Show sample rows to verify.

In [ ]:
for col in ['customerID', 'SupportTicketID', 'ActiveCountryCode']:
    print(f"{col}: {df[col].nunique()} unique values out of {len(df)} rows")

df = df.drop(columns=['customerID', 'SupportTicketID', 'ActiveCountryCode'])
df.sample(5, random_state=0)

`customerID` has 7,043 unique values for 7,043 rows — it is a pure row identifier with a one-to-one mapping to each customer, so it carries no information about churn behavior and would only let a model memorize/overfit to individual rows. `SupportTicketID` similarly has 7,042 unique values across 7,043 rows — essentially unique per customer — so it behaves like another identifier rather than a meaningful predictor. `ActiveCountryCode` has only 1 unique value ("+1") across every row, meaning it has zero variance and cannot help distinguish churners from non-churners. All three columns are dropped, as shown above.

### II. Clean the TotalCharges column (4 pts)
`TotalCharges` is stored as text because a few rows are blank. Show that it is stored as text and how many blanks exist, convert it to numeric, then handle the resulting missing values (these rows correspond to brand new customers). Confirm the column is now numeric.

In [ ]:
print("dtype of TotalCharges:", df['TotalCharges'].dtype)
print("Number of blank/whitespace entries:", (df['TotalCharges'].astype(str).str.strip() == '').sum())

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("\nNumber of missing values after conversion:", df['TotalCharges'].isnull().sum())

# These blanks correspond to brand-new customers (tenure == 0), who have not yet been billed a total amount.
print("\ntenure values for the rows with missing TotalCharges:")
print(df.loc[df['TotalCharges'].isnull(), 'tenure'].unique())

df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("\ndtype of TotalCharges after cleaning:", df['TotalCharges'].dtype)
print("Remaining missing values:", df['TotalCharges'].isnull().sum())

### III. Education level (4 pts = 1 + 2 + 1)

#### 1. Show the unique values of `Education`. (1 pt)

In [ ]:
print(df['Education'].unique())
print(df['Education'].value_counts(dropna=False))

#### 2. Convert `Education` to ordinal values (High School Graduate → 0, Associate Degree → 1, Bachelor's Degree → 2, Master's Degree → 3). This column has many missing values, so decide how to handle them and, in a markdown cell, justify your choice. (2 pts)

In [ ]:
education_map = {
    'High School Graduate': 0,
    'Associate Degree': 1,
    "Bachelor's Degree": 2,
    "Master's Degree": 3
}

df['Education'] = df['Education'].map(education_map)
print("Missing Education values:", df['Education'].isnull().sum(), "out of", len(df))

# Encode missing values as a distinct category (-1) rather than imputing an existing level (see justification below)
df['Education'] = df['Education'].fillna(-1)
print(df['Education'].value_counts().sort_index())

Roughly 80% of the `Education` values are missing (5,679 out of 7,043 rows), which is far too many to safely drop or to impute with the mean/median/mode without introducing significant bias — doing so would fabricate an education level for the vast majority of customers. Since a customer's education level is not reliably inferable from the other features, we instead encode missing values as a distinct category (`-1`) rather than imputing a specific existing category (0-3). This preserves the information that education was not reported, treats "unknown" as its own ordinal level below "High School Graduate," and avoids injecting a false signal into models that would otherwise assume the imputed value was a real observation.

#### 3. Show sample rows to verify the change. (1 pt)

In [ ]:
df[['Education']].sample(10, random_state=0)

### IV. Contract column (4 pts = 1 + 2 + 1)

#### 1. Show the unique values of `Contract`. (1 pt)

In [ ]:
print(df['Contract'].unique())

#### 2. Convert `Contract` to ordinal values based on contract length (Month-to-month → 0, One year → 1, Two year → 2). (2 pts)

In [ ]:
contract_map = {
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
}
df['Contract'] = df['Contract'].map(contract_map)

#### 3. Show the updated unique values. (1 pt)

In [ ]:
print(df['Contract'].unique())
print(df['Contract'].value_counts().sort_index())

### V. Other columns (8 pts = 2 + 4 + 1 + 1)

#### 1. Show the unique values of the remaining nominal columns: `gender`, `Partner`, `Dependents`, `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `PaperlessBilling`, `PaymentMethod`. Storing them in a list and looping is an easy approach. (2 pts)

In [ ]:
nominal_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod']

for col in nominal_cols:
    print(f"{col}: {df[col].unique()}")

#### 2. Use pandas' `get_dummies` to create binary columns for those nominal columns. (4 pts)

In [ ]:
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# get_dummies() creates boolean columns in recent pandas versions; cast them to int for clarity
dummy_cols = [c for c in df.select_dtypes(include='bool').columns]
df[dummy_cols] = df[dummy_cols].astype(int)

print(df.shape)
df.head()

#### 3. In a markdown cell, write a short paragraph on the difference between `get_dummies` and `OneHotEncoder`. (1 pt)

`pandas.get_dummies` is a quick, DataFrame-native convenience function that directly transforms categorical columns into 0/1 indicator columns and returns a new DataFrame with human-readable column names (e.g., `gender_Male`). It is easy to use for one-off exploratory work, but it doesn't "remember" the categories it saw — if the training and test sets are encoded separately and one set is missing a category, the resulting columns can mismatch, and it has no built-in `.transform()`/`.inverse_transform()` API. `sklearn.preprocessing.OneHotEncoder` is a fitted transformer: you call `.fit()` on the training data to learn the categories, then use `.transform()` on any other dataset (including new/test data) to guarantee consistent columns, and it integrates directly into scikit-learn Pipelines and ColumnTransformers, which is important for reproducible, leakage-free machine learning workflows. In short, `get_dummies` is simpler for quick pandas-only work, while `OneHotEncoder` is the more robust, production-appropriate choice when the same encoding must be reliably applied across multiple data splits.

#### 4. Show the first 5 and last 5 rows, and show the shape of the table. (1 pt)

In [ ]:
display(df.head())
display(df.tail())
print(df.shape)

### VI. Encode the target (2 pts)
Convert `Churn` to 1 and 0. Confirm there are no leftover identifier or duplicate raw columns (the columns replaced by `get_dummies` should no longer appear in raw form).

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df['Churn'].value_counts())

# Confirm no leftover identifier columns or raw (pre-dummy) categorical columns remain
print("\nColumns:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes.value_counts())

### VII. Feature scaling (5 pts = 2.5 + 2.5)

#### 1. Use `sklearn.preprocessing.MinMaxScaler` to scale all columns. (2.5 pts)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)
df = df_scaled

#### 2. Show some scaled records. (2.5 pts)

In [ ]:
df.sample(5, random_state=0)

### VIII. Move the target (1 pt)
Move the `Churn` column to the last position of the DataFrame and show that it has changed.

In [ ]:
print("Before:", df.columns.tolist()[-3:])

churn_col = df.pop('Churn')
df['Churn'] = churn_col

print("After:", df.columns.tolist()[-3:])
df.head()

## C. X/Y and Training/Test Split with Stratified Sampling and SMOTE [15 pts = 2 + 1.5 + 4 + 1.5 + 4 + 2]

### I. Copy all features into X and the target into Y. (2 pts)

In [ ]:
X = df.drop(columns=['Churn']).copy()
Y = df['Churn'].copy()
print(X.shape, Y.shape)

### II. Show the ratio of 1 and 0 in Y. (1.5 pts)

In [ ]:
print(Y.value_counts())
print(Y.value_counts(normalize=True).mul(100).round(2))

### III. Use sklearn's `train_test_split` to split the data into training and test sets. The test set should contain 30% of the instances and you must use `random_state = 0`. Use the `stratify` parameter so the class ratio is preserved in both sets. (4 pts)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.30, random_state=0, stratify=Y
)
print("X_train:", X_train.shape, " X_test:", X_test.shape)

### IV. Show the ratio of 1 and 0 in y_train and then y_test. (1.5 pts)

In [ ]:
print("y_train distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\ny_test distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True).mul(100).round(2))

### V. Use imblearn's SMOTE to balance X_train. Keep a copy of the unbalanced training set; you will need it in later sections. (4 pts)

In [ ]:
from imblearn.over_sampling import SMOTE

# Keep a copy of the original (unbalanced) training data for later sections
X_train_unbalanced = X_train.copy()
y_train_unbalanced = y_train.copy()

smote = SMOTE(random_state=0)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(X_train_balanced.shape)

### VI. Show the ratio of 0 and 1 in the rebalanced training set. Consider, do you now have 50% of each class? (2 pts)

In [ ]:
print(y_train_balanced.value_counts())
print(y_train_balanced.value_counts(normalize=True).mul(100).round(2))

Yes — after applying SMOTE, `y_train_balanced` now has an equal number of `0` and `1` labels (50%/50%), because SMOTE generates synthetic minority-class ("Yes"/1) samples until the class counts match the majority class.

## D. PCA and Logistic Regression [20 pts = 7 + 4 + 2 + 2 + 2 + 3]

### I. Build a pipeline to determine how many PCA components give the best logistic regression model, using the balanced training set. The number of components should range up to the maximum number of features you have. Produce a plot and in a markdown cell decide how many components to keep. (7 pts)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

n_features = X_train_balanced.shape[1]
component_range = range(1, n_features + 1)
mean_scores = []

for n in component_range:
    pipe = Pipeline([
        ('pca', PCA(n_components=n, random_state=0)),
        ('lr', LogisticRegression(max_iter=1000, random_state=0))
    ])
    scores = cross_val_score(pipe, X_train_balanced, y_train_balanced, cv=5, scoring='accuracy')
    mean_scores.append(scores.mean())

plt.figure(figsize=(10, 5))
plt.plot(list(component_range), mean_scores, marker='o')
plt.xlabel('Number of PCA Components')
plt.ylabel('Mean CV Accuracy')
plt.title('Logistic Regression Accuracy vs Number of PCA Components')
plt.grid(True)
plt.show()

best_n = list(component_range)[int(np.argmax(mean_scores))]
print(f"Best number of components by CV accuracy: {best_n} (accuracy = {max(mean_scores):.4f})")

Based on the plot, cross-validated accuracy rises quickly for the first several components and then levels off, with only marginal gains (and some noise) as more components are added. We choose `best_n` — the number of components identified by `np.argmax` in the previous cell — because it captures essentially all of the achievable accuracy while meaningfully reducing dimensionality compared to using all original features, keeping the downstream model simpler and less prone to overfitting on noisy/redundant dummy-variable dimensions.

### II. Based on the chosen number of components, evaluate accuracy on the test set using `accuracy_score`. (4 pts)

In [ ]:
from sklearn.metrics import accuracy_score

final_pipe = Pipeline([
    ('pca', PCA(n_components=best_n, random_state=0)),
    ('lr', LogisticRegression(max_iter=1000, random_state=0))
])
final_pipe.fit(X_train_balanced, y_train_balanced)

y_pred = final_pipe.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy with {best_n} PCA components: {test_accuracy:.4f}")

### III. Show the confusion matrix, and in a markdown cell, interpret the numbers you see. (2 pts)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression with PCA')
plt.show()

print(cm)

The confusion matrix shows the counts of true negatives (top-left, correctly predicted "No Churn"), false positives (top-right, predicted "Churn" but the customer actually stayed), false negatives (bottom-left, predicted "No Churn" but the customer actually churned), and true positives (bottom-right, correctly predicted "Churn"). Because churn is the minority class, the number of false negatives is especially important — every false negative is a customer who was about to leave but was not flagged for a retention offer, directly costing the company revenue. The values printed above should be read with that priority in mind: a good churn model minimizes false negatives without inflating false positives (which would waste retention resources on customers who were never going to churn).

### IV. Show precision, recall, and F1 score on the test set. (2 pts)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

### V. Plot the ROC curve and find the AUC. (2 pts)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

y_proba = final_pipe.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression with PCA')
plt.legend()
plt.show()

print(f"AUC: {auc:.4f}")

### VI. Plot the precision–recall curve for different thresholds. In a markdown cell, discuss the plot. (3 pts)

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, pr_thresholds = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(8, 5))
plt.plot(pr_thresholds, precisions[:-1], label='Precision')
plt.plot(pr_thresholds, recalls[:-1], label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs Decision Threshold')
plt.legend()
plt.grid(True)
plt.show()

As the decision threshold increases from 0 to 1, precision generally increases while recall decreases — the model becomes more conservative about predicting "Churn," so the customers it does flag are more likely to be correct (higher precision), but it misses more actual churners (lower recall). The crossover point of the two curves is often a reasonable default operating threshold, but the actual choice should depend on business costs: if the cost of missing a churner (false negative) is higher than the cost of a wasted retention offer (false positive), we should choose a lower threshold that favors recall, and vice versa.

## E. Softmax Regression [3 pts = 2 + 1]

### I. Short answer (markdown): How is softmax regression related to logistic regression? Discuss two different libraries that can build a softmax regression model. (2 pts)

Softmax regression (multinomial logistic regression) is the generalization of (binary) logistic regression to more than two classes. Where logistic regression uses the sigmoid function to output a single probability for a binary outcome, softmax regression uses the softmax function to output a probability distribution across K classes that sums to 1, with its own weight vector (and bias) per class. When K = 2, softmax regression reduces mathematically to standard logistic regression. Two libraries that can build a softmax regression model are: (1) **scikit-learn**, via `sklearn.linear_model.LogisticRegression` with `multi_class='multinomial'` (or the default, which automatically uses multinomial/softmax when there are more than two classes and the solver supports it), and (2) **TensorFlow/Keras**, by building a dense output layer with a `softmax` activation function and training with categorical cross-entropy loss.

### II. Short answer (markdown): If we have 4 classes and 10 features, how many weights plus biases are part of the softmax model? Show your calculation. (1 pt)

For a softmax model with 4 classes and 10 features, each class has its own weight vector of length 10 plus one bias term, i.e., 11 parameters per class. With 4 classes: 4 × (10 + 1) = **44 total weights + biases** (equivalently: 4×10 = 40 weights, plus 4 biases = 44).

## F. KNN [25 pts = 4 + 4 + 3 + 2 + 4 + 4 + 4]

*Always use the rebalanced training set for training unless a section says otherwise.*

### I. Train a KNN classifier (k = 10) on the **unbalanced** training set, test it, and show the confusion matrix and classification report. (4 pts)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_unbalanced = KNeighborsClassifier(n_neighbors=10)
knn_unbalanced.fit(X_train_unbalanced, y_train_unbalanced)
y_pred_unbal = knn_unbalanced.predict(X_test)

print("Confusion Matrix (KNN, k=10, unbalanced training set):")
print(confusion_matrix(y_test, y_pred_unbal))
print()
print(classification_report(y_test, y_pred_unbal, target_names=['No Churn', 'Churn']))

### II. Train a KNN classifier (k = 10) on the **rebalanced** training set, test it, and show the confusion matrix and classification report. (4 pts)

In [ ]:
knn_balanced = KNeighborsClassifier(n_neighbors=10)
knn_balanced.fit(X_train_balanced, y_train_balanced)
y_pred_bal = knn_balanced.predict(X_test)

print("Confusion Matrix (KNN, k=10, rebalanced training set):")
print(confusion_matrix(y_test, y_pred_bal))
print()
print(classification_report(y_test, y_pred_bal, target_names=['No Churn', 'Churn']))

### III. Use grid search to tune the following hyperparameters: number of neighbors (1 to 20), weights (uniform or distance), and metric (Euclidean, Manhattan, or Minkowski). Use multiple evaluation metrics such as AUC and accuracy. (3 pts)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': list(range(1, 21)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=param_grid,
    scoring=['accuracy', 'roc_auc'],
    refit='roc_auc',
    cv=5,
    n_jobs=-1
)

knn_grid.fit(X_train_balanced, y_train_balanced)
print("Grid search complete.")

### IV. Print the best parameters. (2 pts)

In [ ]:
print("Best parameters:", knn_grid.best_params_)
print("Best ROC AUC (CV):", knn_grid.best_score_)

### V. Train a model with the best parameters, test it, and print the confusion matrix, classification report, and the ROC AUC. (4 pts)

In [ ]:
best_knn = KNeighborsClassifier(**knn_grid.best_params_)
best_knn.fit(X_train_balanced, y_train_balanced)
y_pred_best_knn = best_knn.predict(X_test)
y_proba_best_knn = best_knn.predict_proba(X_test)[:, 1]

print("Confusion Matrix (Best KNN):")
print(confusion_matrix(y_test, y_pred_best_knn))
print()
print(classification_report(y_test, y_pred_best_knn, target_names=['No Churn', 'Churn']))
print("ROC AUC:", roc_auc_score(y_test, y_proba_best_knn))

### VI. Apply PCA with the best KNN setup, test it, and print the confusion matrix, classification report, and the ROC AUC. (4 pts)

In [ ]:
pca_knn_pipe = Pipeline([
    ('pca', PCA(n_components=best_n, random_state=0)),
    ('knn', KNeighborsClassifier(**knn_grid.best_params_))
])
pca_knn_pipe.fit(X_train_balanced, y_train_balanced)

y_pred_pca_knn = pca_knn_pipe.predict(X_test)
y_proba_pca_knn = pca_knn_pipe.predict_proba(X_test)[:, 1]

print("Confusion Matrix (PCA + Best KNN):")
print(confusion_matrix(y_test, y_pred_pca_knn))
print()
print(classification_report(y_test, y_pred_pca_knn, target_names=['No Churn', 'Churn']))
print("ROC AUC:", roc_auc_score(y_test, y_proba_pca_knn))

### VII. Short answer (markdown): Write a short discussion of the four KNN models above and their differences. (4 pts)

The four KNN models illustrate the effects of class balance, hyperparameter tuning, and dimensionality reduction. The **unbalanced KNN (k=10)** tends to have high overall accuracy but low recall on the churn class, because with imbalanced training data most neighborhoods are dominated by non-churn points, biasing predictions toward "No Churn." The **rebalanced KNN (k=10, trained on SMOTE-balanced data)** typically improves recall on the churn class (catching more true churners) at some cost to precision/overall accuracy, since it is no longer biased toward the majority class. The **grid-search-tuned KNN** (best `k`, weighting, and distance metric, chosen by CV ROC AUC) generally further improves the balance between precision and recall versus the untuned rebalanced model, since it selects the neighbor count/weighting/metric combination best suited to the data's structure. Finally, applying **PCA before the tuned KNN** reduces the feature space to the `best_n` components identified in Section D; this can slightly change performance — sometimes marginally better because noisy/redundant dummy-variable dimensions are removed and distance calculations (which KNN relies on directly) become less susceptible to the curse of dimensionality, and sometimes marginally worse if useful discriminative information was discarded along with the noise. Overall, class balancing and hyperparameter tuning tend to matter more for KNN's churn-class recall than PCA does in this dataset.